# R04 — Surrogate diagnostics: what does R² = 0.995 actually measure?

**CPU only, ~5–10 min, no images needed.** Addresses reviewer comments
**R2.2** (feature importance / SHAP, R² without the review budget, R² within review-budget bands),
**R1.1** (why gradient-boosted trees rather than a Gaussian-process surrogate; calibrated uncertainty),
**R1.2** (scale-normalised error such as relative RMSE) and the selection-conditioned-fidelity argument of §4.7
(error as a function of the predicted value, i.e. winner's-curse bias).

Protocol notes
* The development split is reproduced exactly (80/20, `random_state=42`, 10 quantile bins for stratification; Proditec
  additionally isolates zero-valued trials) and verified against the committed `X_test.csv` (MVTec) and against the
  test R² stored in `*_surrogate_metadata.json` (both datasets). The **saved** CatBoost model is evaluated unchanged.
* All ablations are re-trained here with the notebook-03 hyper-parameters, but **early stopping uses an inner
  validation split of the training data** (the original notebook early-stops on the test set, a mild optimistic
  leak). The full model is re-trained under the same clean protocol so ablations are compared like-for-like.

In [ ]:
# ============================================================
# C0) Common setup (identical in every review notebook)
# ============================================================
from pathlib import Path
import json, re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
try:
    display
except NameError:
    display = print

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "font.family": "serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "figure.dpi": 150, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
})
SINGLE_COL = (3.35, 2.35)
WIDE_COL = (6.85, 2.65)

# ------------------------------------------------------------
# Robust repo-root detection (CWD, its parents, then rglob)
# ------------------------------------------------------------
CWD = Path.cwd().resolve()


def _is_repo(p: Path) -> bool:
    return (p / "mvtec" / "outputs_patchcore").is_dir() and (p / "proditec" / "outputs_patchcore").is_dir()


REPO = next((p for p in [CWD, *CWD.parents] if _is_repo(p)), None)
if REPO is None:
    for base in [CWD, CWD.parent]:
        for hit in base.rglob("outputs_patchcore"):
            if _is_repo(hit.parent.parent):
                REPO = hit.parent.parent
                break
        if REPO:
            break
if REPO is None:
    raise FileNotFoundError("Could not locate the repository root (needs mvtec/ and proditec/). "
                            "Launch Jupyter from the repo or from review/.")
REVIEW = REPO / "review"
OUT = REVIEW / "outputs"
OUT_TABLES, OUT_FIGS = OUT / "tables", OUT / "figures"
for d in (OUT_TABLES, OUT_FIGS):
    d.mkdir(parents=True, exist_ok=True)

DS = {
    "mvtec": dict(root=REPO / "mvtec", pre="mvtec",
                  data="surrogate_ready_dataset/mvtec_4class_patchcore_surrogate_dataset.csv"),
    "proditec": dict(root=REPO / "proditec", pre="proditec",
                     data="surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv"),
}
for k, v in DS.items():
    r = v["root"]
    v["art"] = r / "surrogate_pkl_cfs_metadata"
    v["cfs"] = v["art"] / f"{k}_counterfactuals" / f"{k}_all_methods_cfs.csv"
    v["eval"] = v["art"] / f"{k}_counterfactuals_evaluation" / f"{k}_all_methods_cfs_patchcore_evaluation.csv"
    v["rev"] = v["art"] / f"{k}_revision_analysis"
    v["decomp"] = r / "revision_outputs" / "tables" / f"decomposition_per_candidate_{k}.csv"
    v["model"] = v["art"] / f"{k}_surrogate_catboost.cbm"
    v["meta"] = v["art"] / f"{k}_surrogate_metadata.json"
    v["dataset"] = r / v["data"]
    v["confirm"] = r / "revision_outputs" / "tables" / f"baseline_confirmation_{k}.csv"
    v["confirm_cands"] = r / "revision_outputs" / "tables" / f"baseline_confirm_candidates_{k}.csv"
print("REPO  :", REPO)
print("OUT   :", OUT)


def save_table(table: pd.DataFrame, stem: str, ff="%.4f"):
    p = OUT_TABLES / f"{stem}.csv"
    table.to_csv(p, index=False)
    try:
        table.to_latex(OUT_TABLES / f"{stem}.tex", index=False, escape=True, float_format=ff)
    except Exception as e:
        print("  (tex export skipped:", e, ")")
    print("Saved table:", p.relative_to(REPO))
    return p


def save_fig(fig, stem: str):
    for ext in ("pdf", "png"):
        fig.savefig(OUT_FIGS / f"{stem}.{ext}", bbox_inches="tight")
    plt.close(fig)
    print("Saved figure:", (OUT_FIGS / f"{stem}.pdf").relative_to(REPO))


def merge_numbers(section: str, values: dict):
    """Append to outputs/frozen_numbers.json (one section per notebook)."""
    p = OUT / "frozen_numbers.json"
    cur = json.loads(p.read_text()) if p.exists() else {}
    cur[section] = json.loads(json.dumps(values, default=float))
    p.write_text(json.dumps(cur, indent=2, sort_keys=True))
    print("Updated", p.relative_to(REPO), "->", section)

In [ ]:
# ============================================================
# R4.1) Imports, data, split reproduction, saved surrogate
# ============================================================
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr

RUN_GP = True          # Gaussian-process comparison (R1.1); set False to skip (~2-4 min)
GP_MAX_TRAIN = 1500    # exact GP is O(n^3): subsample the training set
SOFT = ["params_soft_review_budget", "params_soft_train_fraction", "params_soft_corruption_level"]
BUDGET = "params_soft_review_budget"
CB_KW = dict(loss_function="RMSE", iterations=5000, learning_rate=0.05, depth=8, l2_leaf_reg=3.0,
             random_seed=SEED, verbose=False, thread_count=-1)


def rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))


S = {}
for ds, s in DS.items():
    meta = json.loads(s["meta"].read_text())
    feats, cats = meta["features"], meta["categorical_features"]
    nums = [c for c in feats if c not in cats]
    df = pd.read_csv(s["dataset"])
    X, y = df[feats].copy(), df["value"].copy()
    for c in cats:
        X[c] = X[c].astype(str)
    if ds == "proditec":   # proditec/03: zero-valued F1 trials form their own stratum, the rest 10 quantile bins
        ys_ = y.reset_index(drop=True); z = ys_ == 0.0
        strat = pd.Series(index=ys_.index, dtype="object"); strat[z] = "zero"
        strat.loc[ys_[~z].index] = pd.qcut(ys_[~z], q=10, duplicates="drop").astype(str)
    else:                  # mvtec/03: 10 quantile bins
        strat = pd.qcut(y, 10, labels=False, duplicates="drop")
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, random_state=SEED, shuffle=True, stratify=strat)
    saved = CatBoostRegressor(); saved.load_model(str(s["model"]))
    cols = list(saved.feature_names_)
    p = np.asarray(saved.predict(Xte[cols]), float)
    S[ds] = dict(meta=meta, feats=feats, cats=cats, nums=nums, Xtr=Xtr, Xte=Xte, ytr=ytr, yte=yte, saved=saved,
                 cols=cols, pred_saved=p)
    msg = ""
    if ds == "mvtec":
        xt = pd.read_csv(s["art"] / "mvtec_train_test_split" / "X_test.csv", index_col=0)
        msg = f" | split identical to committed X_test.csv: {set(Xte.index) == set(xt.index)}"
    print(f"{ds}: n={len(df)} train={len(Xtr)} test={len(Xte)} | saved model test R2={r2_score(yte, p):.4f} "
          f"(metadata: {meta['r2']:.4f}){msg}")

In [ ]:
# ============================================================
# R4.2) Scale-normalised errors (R1.2) for the SAVED surrogate
# ============================================================
rows = []
for ds, d in S.items():
    y, p = d["yte"].to_numpy(), d["pred_saved"]
    r = rmse(y, p)
    rows.append(dict(dataset=ds, n_test=len(y), R2=r2_score(y, p), RMSE=r, MAE=mean_absolute_error(y, p),
                     y_mean=y.mean(), y_sd=y.std(), y_range=y.max() - y.min(),
                     rRMSE_mean=r / y.mean(), rRMSE_sd=r / y.std(), NRMSE_range=r / (y.max() - y.min()),
                     rMAE_mean=mean_absolute_error(y, p) / y.mean(),
                     MAPE=float(np.mean(np.abs(y - p) / np.clip(y, 1e-3, None)))))
REL = pd.DataFrame(rows)
display(REL.round(4))
save_table(REL.round(5), "R04_relative_error_metrics")

In [ ]:
# ============================================================
# R4.3) How much of the target does the review budget carry?  (R2.2)
#   models (all re-trained, inner-validation early stopping):
#     full | no review budget | hard hyper-parameters only | review budget only
# ============================================================
def fit_cb(Xtr, ytr, feats, cats):
    Xa, Xv, ya, yv = train_test_split(Xtr[feats], ytr, test_size=0.1, random_state=SEED)
    m = CatBoostRegressor(**CB_KW)
    m.fit(Xa, ya, cat_features=[feats.index(c) for c in cats if c in feats], eval_set=(Xv, yv),
          use_best_model=True, early_stopping_rounds=200)
    return m


ABL = {}
for ds, d in S.items():
    sets = {
        "full (13 features)": d["feats"],
        "without review budget": [f for f in d["feats"] if f != BUDGET],
        "hard hyper-parameters only": [f for f in d["feats"] if f not in SOFT],
        "review budget only": [BUDGET],
        "soft parameters only": SOFT,
    }
    for name, fs in sets.items():
        cs = [c for c in d["cats"] if c in fs]
        m = fit_cb(d["Xtr"], d["ytr"], fs, cs)
        p = np.asarray(m.predict(d["Xte"][fs]), float)
        ABL[(ds, name)] = dict(model=m, feats=fs, pred=p)
        print(f"{ds:9s} {name:28s} R2={r2_score(d['yte'], p):.4f}  RMSE={rmse(d['yte'], p):.4f}  iters={m.get_best_iteration()}")

rows = [dict(dataset=ds, model=name, n_features=len(v["feats"]), R2=r2_score(S[ds]["yte"], v["pred"]),
             RMSE=rmse(S[ds]["yte"], v["pred"]), MAE=mean_absolute_error(S[ds]["yte"], v["pred"]),
             rRMSE_mean=rmse(S[ds]["yte"], v["pred"]) / S[ds]["yte"].mean())
        for (ds, name), v in ABL.items()]
AB = pd.DataFrame(rows)
# share of variance NOT explained by the budget that the remaining features explain
for ds in S:
    r_full = AB[(AB.dataset == ds) & (AB.model == "full (13 features)")].R2.iloc[0]
    r_b = AB[(AB.dataset == ds) & (AB.model == "review budget only")].R2.iloc[0]
    AB.loc[(AB.dataset == ds), "partial_R2_beyond_budget"] = (r_full - r_b) / (1 - r_b)
display(AB.round(4))
save_table(AB.round(5), "R04_ablation_r2")

In [ ]:
# ============================================================
# R4.4) R² within review-budget bands  (R2.2)  — test set, SAVED surrogate and budget-free model
# ============================================================
rows = []
for ds, d in S.items():
    b = d["Xte"][BUDGET].astype(float).to_numpy()
    y = d["yte"].to_numpy()
    edges = np.unique(np.quantile(b, np.linspace(0, 1, 6)))
    band = np.clip(np.searchsorted(edges, b, side="right") - 1, 0, len(edges) - 2)
    for k in range(len(edges) - 1):
        m = band == k
        if m.sum() < 10:
            continue
        for name, p in [("saved surrogate", d["pred_saved"]),
                        ("without review budget", ABL[(ds, "without review budget")]["pred"])]:
            rows.append(dict(dataset=ds, band=f"[{edges[k]:.3f}, {edges[k + 1]:.3f}]", n=int(m.sum()), model=name,
                             y_sd_in_band=y[m].std(), RMSE=rmse(y[m], p[m]), R2_within_band=r2_score(y[m], p[m]),
                             spearman_within_band=spearmanr(y[m], p[m]).statistic))
BAND = pd.DataFrame(rows)
display(BAND.round(3))
save_table(BAND.round(4), "R04_r2_within_budget_bands")

fig, axes = plt.subplots(1, 2, figsize=WIDE_COL, sharey=True)
for ax, ds in zip(axes, S):
    t = BAND[(BAND.dataset == ds) & (BAND.model == "saved surrogate")]
    ax.bar(range(len(t)), t.R2_within_band, color="#3b82d4")
    ax.axhline(S[ds]["meta"]["r2"], color="k", lw=0.8, ls="--", label="global R² (saved)")
    ax.set_xticks(range(len(t))); ax.set_xticklabels([b.replace(", ", "–") for b in t.band], rotation=30, ha="right")
    ax.set_title(ds); ax.set_xlabel("review-budget band"); ax.legend(frameon=False)
axes[0].set_ylabel("R² within band")
save_fig(fig, "R04_r2_within_budget_bands")

In [ ]:
# ============================================================
# R4.4b) The direct answer to "how much do the HARD hyper-parameters tell us?":
#        fit hard-parameters-only models INSIDE each review-budget band (budget is ~constant there).
#        A negative R² above for the budget-free model is by construction (it never sees the budget);
#        this table is the fair within-band test.
# ============================================================
rows = []
for ds, d in S.items():
    hard = [f for f in d["feats"] if f not in SOFT]
    soft_other = [f for f in SOFT if f != BUDGET]
    btr, bte = d["Xtr"][BUDGET].astype(float).to_numpy(), d["Xte"][BUDGET].astype(float).to_numpy()
    edges = np.unique(np.quantile(np.r_[btr, bte], np.linspace(0, 1, 6)))
    btr_k = np.clip(np.searchsorted(edges, btr, side="right") - 1, 0, len(edges) - 2)
    bte_k = np.clip(np.searchsorted(edges, bte, side="right") - 1, 0, len(edges) - 2)
    for k in range(len(edges) - 1):
        mtr, mte = btr_k == k, bte_k == k
        if mtr.sum() < 60 or mte.sum() < 20:
            continue
        for name, fs in [("hard only", hard), ("hard + other soft (no budget)", hard + soft_other)]:
            cs = [c for c in d["cats"] if c in fs]
            m = fit_cb(d["Xtr"][mtr], d["ytr"][mtr], fs, cs)
            p = np.asarray(m.predict(d["Xte"][mte][fs]), float)
            y = d["yte"].to_numpy()[mte]
            rows.append(dict(dataset=ds, band=f"[{edges[k]:.3f}, {edges[k + 1]:.3f}]", n_train=int(mtr.sum()), n_test=int(mte.sum()),
                             features=name, R2_within_band=r2_score(y, p), spearman=spearmanr(y, p).statistic, y_sd=y.std()))
BAND2 = pd.DataFrame(rows)
display(BAND2.round(3))
save_table(BAND2.round(4), "R04_within_band_hard_only_models")

In [ ]:
# ============================================================
# R4.5) SHAP values (native CatBoost TreeSHAP) and permutation importance  (R2.2)
# ============================================================
def shap_table(model, X, feats, cats):
    pool = Pool(X[feats], cat_features=[feats.index(c) for c in cats if c in feats])
    sv = model.get_feature_importance(pool, type="ShapValues")      # (n, n_features+1)
    return pd.Series(np.abs(sv[:, :-1]).mean(axis=0), index=feats), sv


def perm_importance(model, X, y, feats, n_rep=5):
    base = r2_score(y, model.predict(X[feats]))
    rng = np.random.default_rng(SEED)
    out = {}
    for f in feats:
        drops = []
        for _ in range(n_rep):
            Xp = X[feats].copy(); Xp[f] = rng.permutation(Xp[f].to_numpy())
            drops.append(base - r2_score(y, model.predict(Xp)))
        out[f] = float(np.mean(drops))
    return pd.Series(out)


IMP = []
fig, axes = plt.subplots(1, 2, figsize=(6.85, 3.0))
for ax, ds in zip(axes, S):
    d = S[ds]
    full = ABL[(ds, "full (13 features)")]
    nob = ABL[(ds, "without review budget")]
    s_full, _ = shap_table(full["model"], d["Xte"], full["feats"], [c for c in d["cats"] if c in full["feats"]])
    s_nob, _ = shap_table(nob["model"], d["Xte"], nob["feats"], [c for c in d["cats"] if c in nob["feats"]])
    pi_full = perm_importance(full["model"], d["Xte"], d["yte"], full["feats"])
    pi_nob = perm_importance(nob["model"], d["Xte"], d["yte"], nob["feats"])
    t = pd.DataFrame({"mean_abs_shap_full": s_full, "perm_R2drop_full": pi_full,
                      "mean_abs_shap_without_budget": s_nob, "perm_R2drop_without_budget": pi_nob})
    t["share_shap_full"] = t.mean_abs_shap_full / t.mean_abs_shap_full.sum()
    t["share_shap_without_budget"] = t.mean_abs_shap_without_budget / t.mean_abs_shap_without_budget.sum()
    t["kind"] = np.where(t.index.isin(SOFT), "soft (operator-facing)", "hard (model)")
    t.insert(0, "dataset", ds); t = t.reset_index(names="feature").sort_values("mean_abs_shap_full", ascending=False)
    IMP.append(t)
    tt = t.set_index("feature").mean_abs_shap_full.sort_values()
    ax.barh([f.replace("params_", "") for f in tt.index], tt.values,
            color=["#7c5cd8" if f in SOFT else "#3b82d4" for f in tt.index])
    ax.set_title(f"{ds}: mean |SHAP| (full surrogate)"); ax.set_xlabel("mean |SHAP| (F1 units)")
IMP = pd.concat(IMP, ignore_index=True)
display(IMP.round(4))
save_table(IMP.round(5), "R04_feature_importance_shap_permutation")
fig.tight_layout(); save_fig(fig, "R04_shap_importance")

In [ ]:
# ============================================================
# R4.6) Winner's curse: signed error as a function of the PREDICTED value (test set, saved surrogate)
#       — the surrogate-level analogue of "selection-conditioned fidelity"
# ============================================================
rows = []
for ds, d in S.items():
    p, y = d["pred_saved"], d["yte"].to_numpy()
    q = np.quantile(p, [0, .5, .75, .9, .95, 1.0])
    for lo, hi, lab in [(q[0], q[1], "bottom 50 %"), (q[1], q[2], "50–75 %"), (q[2], q[3], "75–90 %"),
                        (q[3], q[4], "90–95 %"), (q[4], q[5] + 1e-9, "top 5 %")]:
        m = (p >= lo) & (p < hi)
        rows.append(dict(dataset=ds, predicted_quantile_band=lab, n=int(m.sum()), mean_pred=p[m].mean(), mean_recorded=y[m].mean(),
                         mean_signed_error_pred_minus_recorded=(p[m] - y[m]).mean(), RMSE=rmse(y[m], p[m])))
WC = pd.DataFrame(rows)
display(WC.round(4))
save_table(WC.round(5), "R04_winners_curse_by_predicted_quantile")

In [ ]:
# ============================================================
# R4.7) Gaussian-process surrogate vs CatBoost (R1.1): accuracy AND calibrated uncertainty
#       Same training subsample for both; one-hot categoricals; Matern-5/2 ARD-free + white noise.
#       Honest scope: exact GPs do not scale to the full 8k-row MVTec log; this is a feasibility comparison.
# ============================================================
GPR = []
if RUN_GP:
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
    from sklearn.preprocessing import StandardScaler
    from scipy.stats import norm
    import time as _t
    for ds, d in S.items():
        rng = np.random.default_rng(SEED)
        idx = rng.choice(len(d["Xtr"]), size=min(GP_MAX_TRAIN, len(d["Xtr"])), replace=False)
        Xs, ys = d["Xtr"].iloc[idx], d["ytr"].iloc[idx]
        enc = lambda X: pd.get_dummies(X[d["feats"]], columns=d["cats"]).astype(float)
        Etr, Ete = enc(pd.concat([Xs, d["Xte"]])).iloc[:len(Xs)], enc(pd.concat([Xs, d["Xte"]])).iloc[len(Xs):]
        sc = StandardScaler().fit(Etr)
        t0 = _t.time()
        gp = GaussianProcessRegressor(C(1.0) * Matern(length_scale=3.0, nu=2.5) + WhiteKernel(1e-2),
                                      normalize_y=True, n_restarts_optimizer=0, random_state=SEED)
        gp.fit(sc.transform(Etr), ys.to_numpy())
        mu, sd = gp.predict(sc.transform(Ete), return_std=True)
        t_gp = _t.time() - t0
        cb = fit_cb(Xs, ys, d["feats"], d["cats"])
        pc = np.asarray(cb.predict(d["Xte"][d["feats"]]), float)
        y = d["yte"].to_numpy()
        z90 = norm.ppf(0.95)
        err = np.abs(y - mu)
        GPR.append(dict(dataset=ds, n_train_used=len(Xs), GP_R2=r2_score(y, mu), GP_RMSE=rmse(y, mu),
                        CatBoost_same_subsample_R2=r2_score(y, pc), CatBoost_same_subsample_RMSE=rmse(y, pc),
                        CatBoost_full_train_R2=r2_score(y, d["pred_saved"]),
                        GP_90pct_interval_coverage=float(np.mean(err <= z90 * sd)),
                        spearman_predicted_sd_vs_abs_error=float(spearmanr(sd, err).statistic),
                        GP_fit_predict_sec=t_gp))
    GPR = pd.DataFrame(GPR)
    display(GPR.round(4))
    save_table(GPR.round(5), "R04_gp_vs_catboost")
else:
    print("RUN_GP=False -> skipped")

In [ ]:
# ============================================================
# R4.8) Freeze
# ============================================================
nums = dict(
    relative_error=REL.round(5).to_dict(orient="records"),
    ablation_r2=AB.round(5).to_dict(orient="records"),
    r2_within_band_saved=BAND[BAND.model == "saved surrogate"].groupby("dataset").R2_within_band.agg(["min", "median", "max"]).round(4)
        .reset_index().to_dict(orient="records"),
    shap_top3=IMP.sort_values("mean_abs_shap_full", ascending=False).groupby("dataset").head(3)[["dataset", "feature", "share_shap_full"]]
        .round(4).to_dict(orient="records"),
    within_band_hard_only=BAND2[BAND2.features == "hard only"].groupby("dataset").R2_within_band.agg(["min", "median", "max"]).round(4)
        .reset_index().to_dict(orient="records"),
    winners_curse=WC.round(5).to_dict(orient="records"),
    gp=(GPR.round(5).to_dict(orient="records") if len(GPR) else "skipped"),
)
merge_numbers("R04_surrogate_diagnostics", nums)